# Flexible Multitask Computation in RNNs — Paradigm A

This notebook reproduces the core analyses of

> Driscoll, L. N., Shenoy, K., & Sussillo, D. (2024).
> Flexible multitask computation in recurrent networks utilizes shared dynamical motifs.
> *Nature Neuroscience*, 27, 1349–1363.

The paper asks how a single recurrent network can flexibly reconfigure its dynamics
to perform many related cognitive tasks.  The central idea is the **dynamical motif**:
a recurring pattern of neural activity (e.g. a ring attractor, a decision boundary, a
rotation) that implements a specific computation and is reused across tasks.

The notebook is organized in three parts, matching the paper:

1. **Part I — Single-task network**: train/load a `delaygo` (MemoryPro) network and
   reproduce **Figure 1** (fixed-point skeleton of one task).
2. **Part II — Two-task network**: train/load a `delaygo` + `delayanti`
   (MemoryPro + MemoryAnti) network and reproduce **Figure 2** (rule-input
   bifurcation diagrams show shared fixed points across tasks).
3. **Part III — 15-task network**: train/load the full 15-task network and reproduce
   **Figure 3A** (variance-matrix atlas of dynamical motifs), **Figure 3D/E**
   (shared attractors across task pairs), and **Figure 4B/C** (task geometry in
   state space).

The three parts of RNN training may take several hours. 

## Setup
load all packages, set paths and hyper-parameters, and define helper
functions used across the three parts. You can skip this section and start from Part 1.

In [ ]:
# Global imports and paths
# -------------------------------------------------------------------------
import os
import sys
import warnings
import pickle
import hashlib
import string
import csv

import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib import cm
import matplotlib.patches as mpatches
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from sklearn.decomposition import PCA
from scipy import linalg
from scipy.cluster import hierarchy
from scipy.cluster.hierarchy import fcluster, linkage, dendrogram
from scipy.spatial import distance
from sklearn.metrics import silhouette_score
from adjustText import adjust_text

warnings.filterwarnings("ignore")

# -------------------------------------------------------------------------
# Global constants
# -------------------------------------------------------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOTEBOOK_DIR = os.getcwd()
SRC_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..', 'src'))
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from neuralrnn.models.ctrnn.configuration_ctrnn import CTRNNConfig
from neuralrnn.models.ctrnn.modeling_ctrnn import CTRNNModel
from neuralrnn.train.trainer import Trainer
from neuralrnn.train.training_args import TrainingArguments
from neuralrnn.train.objectives import build_objective
from neuralrnn.analysis.fixed_points import NumericFixedPointFinder, FixedPoint
from neuralrnn.data.tasks.multitask_flexible_task import (
    generate_trials, RULES_ALL, RULE_INDEX_MAP, RULE_NAME,
)
from neuralrnn.data.tasks.multitask_flexible_dataset import MultitaskFlexibleDataset

MODEL_DIR_SINGLE = os.path.join(NOTEBOOK_DIR, "models", "13", "flexible_delaygo_128")
MODEL_DIR_DUAL = os.path.join(NOTEBOOK_DIR, "models", "13", "flexible_delaygo_delayanti_128")
MODEL_DIR_MULTI = os.path.join(NOTEBOOK_DIR, "models", "13", "flexible_multitask")
FIG_DIR = os.path.join(NOTEBOOK_DIR, "figs", "13")

for d in [MODEL_DIR_SINGLE, MODEL_DIR_DUAL, MODEL_DIR_MULTI, FIG_DIR]:
    os.makedirs(d, exist_ok=True)

DT = 20.0
TAU = 100.0
INPUT_DIM = 20
OUTPUT_DIM = 3
LATENT_DIM = 128
RULE_START = 5
N_CONDITIONS = 400
N_ANGLES = 8

# Fixed-point search defaults (from 12b; do not tighten).
FP_BACKEND = "numeric"
N_CANDIDATES = 400
N_ITERS = 5000
LR = 1e-3
SPEED_TOL = 5e-1
NOISE_SCALE = 0.01
Q_THRESH = 5e-2
STABILITY_THRESH = 1.1
DEDUP_TOL = 8e-2

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"DEVICE={DEVICE}")
print(f"MODEL_DIR_SINGLE={MODEL_DIR_SINGLE}")
print(f"MODEL_DIR_DUAL={MODEL_DIR_DUAL}")
print(f"MODEL_DIR_MULTI={MODEL_DIR_MULTI}")
print(f"FIG_DIR={FIG_DIR}")


### Training helpers

`RegularizedSupervisedObjective` implements the masked
mean-squared-error loss used in the paper, with small L2 penalties on activity
and weights.  `build_flexible_model` creates the CTRNN (softplus activation,
recurrent weights initialized to `0.8 I`), and `train_or_load` either loads an
existing checkpoint or trains a new model.

In [ ]:
def build_flexible_model(latent_dim=LATENT_DIM):
    config = CTRNNConfig(
        input_dim=INPUT_DIM,
        latent_dim=latent_dim,
        output_dim=OUTPUT_DIM,
        dt=DT,
        tau=TAU,
        activation="softplus",
        sigma_rec=0.02,
        noise_alpha_scaling=True,
        trainable_h0=False,
    )
    m = CTRNNModel(config).to(DEVICE)
    with torch.no_grad():
        torch.nn.init.eye_(m.h2h.weight)
        m.h2h.weight *= 0.8
        m.input2h.weight.normal_(0, 1.0 / np.sqrt(INPUT_DIM))
        m.readout_layer.bias.zero_()
    return m


def train_or_load(rule_set, save_dir, latent_dim=LATENT_DIM, max_steps=6000, overwrite=False):
    os.makedirs(save_dir, exist_ok=True)
    if os.path.exists(os.path.join(save_dir, "model.safetensors")) and not overwrite:
        print(f"Loading existing checkpoint from {save_dir}")
        m = CTRNNModel.from_pretrained(save_dir)
        m.to(DEVICE)
        return m
    print(f"Training {rule_set} -> {save_dir} ({max_steps} steps, {latent_dim} units)")
    m = build_flexible_model(latent_dim)
    if set(rule_set) == set(RULES_ALL):
        ds = MultitaskFlexibleDataset(
            rules=rule_set,
            rule_prob_map={"contextdelaydm1": 10.0, "contextdelaydm2": 10.0},
            batch_size=512,
            mode="random",
            sigma_x=0.01,
            seed=SEED,
        )
    else:
        ds = MultitaskFlexibleDataset(
            rules=rule_set,
            batch_size=512,
            mode="random",
            sigma_x=0.01,
            seed=SEED,
        )
    #  loss = MSE(masked, global reduction) + l2_h * global mean(states^2) + l2_weight * sum(p^2)
    # Implemented with the framework's reusable RegularizedSupervisedObjective via build_objective.
    # Direct class equivalent:
    #   from neuralrnn.train.objectives import RegularizedSupervisedObjective
    #   obj = RegularizedSupervisedObjective(
    #       task_type="regression", mse_reduce="global", activity_weight=1e-6,
    #       activity_reduce="global", weight_weight=1e-6, weight_reduce="sum",
    #   )
    obj = build_objective(
        "regularized_supervised",
        task_type="regression",
        mse_reduce="global",
        activity_weight=1e-6,
        activity_reduce="global",
        weight_weight=1e-6,
        weight_reduce="sum",
    )
    args = TrainingArguments(
        output_dir=save_dir,
        max_steps=max_steps,
        batch_size=512,
        learning_rate=0.001,
        log_every=500,
        save_every=1000,
        device=str(DEVICE),
        seed=SEED,
    )
    trainer = Trainer(model=m, dataset=ds, objective=obj, args=args)
    trainer.train()
    m.save_pretrained(save_dir)
    return m


### Epoch, input, and forward-pass helpers

These helpers translate between the trial structure returned by
`generate_trials` (a dict of epoch `(start, end)` indices) and the operations we need
for analysis: extracting states for a task period, building the input vector at period
onset, or running the network deterministically.

In [ ]:
def run_forward(model, inputs):
    """Deterministic forward pass; returns (outputs, states) numpy arrays."""
    model.eval()
    with torch.no_grad():
        out = model(torch.from_numpy(inputs).float().to(DEVICE))
    return out.outputs.cpu().numpy(), out.states.cpu().numpy()


def get_epoch_slice(epochs, period, T):
    s, e = epochs.get(period, (0, T))
    if s is None:
        s = 0
    if e is None:
        e = T
    return s, e


def get_period_states(states, epochs, period):
    s, e = get_epoch_slice(epochs, period, states.shape[1])
    return states[:, s:e, :]


def get_period_input_at_onset(inputs, epochs, period):
    s, e = get_epoch_slice(epochs, period, inputs.shape[1])
    if s < inputs.shape[1]:
        return inputs[:, s, :].mean(axis=0)
    return inputs[:, 0, :].mean(axis=0)


def period_input_with_rule(base_input, rule):
    v = base_input.copy()
    v[RULE_START:RULE_START + len(RULES_ALL)] = 0.0
    v[RULE_START + RULE_INDEX_MAP[rule]] = 1.0
    return v


def get_stim_angles(inputs, epochs):
    s, e = get_epoch_slice(epochs, "stim1", inputs.shape[1])
    mod1 = inputs[:, s:e, 1:3]
    angle = np.arctan2(mod1[:, :, 0].mean(axis=1), mod1[:, :, 1].mean(axis=1))
    return np.mod(angle, 2 * np.pi)

### Fixed-point search helpers

Fixed points are found with `NumericFixedPointFinder`.  The key parameters are
deliberately loose (`q_thresh=5e-2`, `dedup_tol=8e-2`, `noise_scale=0.01`) because the
ring attractors in this paper are easier to recover with tolerant thresholds.

The `search_period_fps` function implements the **ring/central strategy** from the
reference notebook (`12b`):

* `fix1` and `delay1` are searched once from the states at the end of the relevant
  period (central fixed point) **and** from states that lie on the ring.
* `stim1` is searched separately for each stimulus angle because the input depends on
  the angle.
* `go1` is seeded from delay-end states and, optionally, from the delay-period fixed
  points.

In [ ]:
def resolve_fp_params(overrides=None):
    """Merge global fixed-point defaults with optional per-period overrides."""
    params = {
        "backend": FP_BACKEND,
        "n_candidates": N_CANDIDATES,
        "n_iters": N_ITERS,
        "lr": LR,
        "speed_tol": SPEED_TOL,
        "noise_scale": NOISE_SCALE,
        "q_thresh": Q_THRESH,
        "stability_thresh": STABILITY_THRESH,
        "dedup_tol": DEDUP_TOL,
        "init_scale": 0.5,
        "init_positive": True,
    }
    if overrides:
        params.update(overrides)
    return params


def make_finder(params):
    """Instantiate the numeric fixed-point finder."""
    backend = params["backend"]
    if backend == "numeric":
        return NumericFixedPointFinder(
            n_candidates=params["n_candidates"],
            n_iters=params["n_iters"],
            lr=params["lr"],
            speed_tol=params["speed_tol"],
            dedup_tol=params["dedup_tol"],
            init_scale=params["init_scale"],
            init_positive=params["init_positive"],
        )
    else:
        raise ValueError(f"Unknown backend={backend}")

def find_fps_for_period(model, task_input, init_states, params=None):
    params = resolve_fp_params(params)
    if init_states is None or len(init_states) == 0:
        return []
    init_states = np.asarray(init_states)
    M = model.config.latent_dim
    n = params["n_candidates"]

    idx = np.random.choice(len(init_states), size=n, replace=True)
    z0 = init_states[idx] + np.random.randn(n, M) * params["noise_scale"]
    z0 = np.maximum(z0, 0.0)

    finder = make_finder(params)
    fps = finder.find(
        model,
        task_input=torch.from_numpy(task_input).float().to(DEVICE),
        init_states=torch.from_numpy(z0).float().to(DEVICE),
    )

    filtered = []
    for fp in fps.points:
        if fp.speed >= params["q_thresh"]:
            continue
        if fp.eigenvalues is None:
            is_stable = True
        else:
            is_stable = bool(np.max(np.abs(fp.eigenvalues)) <= params["stability_thresh"])
        filtered.append(FixedPoint(
            z=fp.z, speed=fp.speed,
            eigenvalues=fp.eigenvalues, is_stable=is_stable,
        ))
    if not filtered and fps.points:
        best = min(fps.points, key=lambda fp: fp.speed)
        is_stable = (best.eigenvalues is None or
                     bool(np.max(np.abs(best.eigenvalues)) <= params["stability_thresh"]))
        filtered = [FixedPoint(
            z=best.z, speed=best.speed,
            eigenvalues=best.eigenvalues, is_stable=is_stable,
        )]
    return filtered


def dedup_fps(fps, tol=1e-2):
    deduped = []
    for fp in fps:
        if not any(np.linalg.norm(fp.z - d.z) < tol for d in deduped):
            deduped.append(fp)
    return deduped

def _angle_mask(stim_angles, theta):
    """Boolean mask of trials whose stim1 angle is close to theta."""
    ang_diff = np.abs(np.angle(np.exp(1j * (stim_angles - theta))))
    return ang_diff <= (np.pi / N_ANGLES + 1e-3)

def search_period_fps(model, inputs, states, epochs, period,
                      params=None, central_seeds=None,
                      angles=None, stim_angles=None, rule="delaygo"):
    params = resolve_fp_params(params)
    if angles is None:
        angles = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
    if stim_angles is None:
        stim_angles = get_stim_angles(inputs, epochs)

    T = states.shape[1]
    s_f, e_f = get_epoch_slice(epochs, "fix1", T)
    s_s, e_s = get_epoch_slice(epochs, "stim1", T)
    s_d, e_d = get_epoch_slice(epochs, "delay1", T)

    fix_end_states = states[:, e_f - 1, :]
    stim_end_states = states[:, e_s - 1, :]
    delay_end_states = states[:, e_d - 1, :]

    u_fix = period_input_with_rule(get_period_input_at_onset(inputs, epochs, "fix1"), rule)
    u_delay = period_input_with_rule(get_period_input_at_onset(inputs, epochs, "delay1"), rule)
    u_go = period_input_with_rule(get_period_input_at_onset(inputs, epochs, "go1"), rule)

    fps_all = []
    angle_fps = {}

    if period == "fix1":
        fps_all.extend(find_fps_for_period(model, u_fix, fix_end_states, params))
        fps_all.extend(find_fps_for_period(model, u_fix, delay_end_states, params))
    elif period == "stim1":
        for theta in angles:
            mask = _angle_mask(stim_angles, theta)
            if mask.sum() == 0:
                continue
            init_states = fix_end_states[mask]
            u = inputs[mask, e_f, :].mean(axis=0)
            u = period_input_with_rule(u, rule)
            fps = find_fps_for_period(model, u, init_states, params)
            angle_fps[theta] = fps
            fps_all.extend(fps)
    elif period == "delay1":
        fps_all.extend(find_fps_for_period(model, u_delay, delay_end_states, params))
        fps_all.extend(find_fps_for_period(model, u_delay, stim_end_states, params))
    elif period == "go1":
        go_central_seeds = delay_end_states
        if central_seeds:
            central_z = np.array([fp.z for fp in central_seeds])
            go_central_seeds = np.concatenate([go_central_seeds, central_z], axis=0)
        fps_all.extend(find_fps_for_period(model, u_go, go_central_seeds, params))
        fps_all.extend(find_fps_for_period(model, u_go, delay_end_states, params))
    else:
        raise ValueError(f"Unknown period={period}")

    fps_all = dedup_fps(fps_all, tol=params.get("dedup_tol", 1e-2))
    return fps_all, angle_fps


def compute_all_period_fps(model, inputs, states, epochs,
                           period_params=None, angles=None, stim_angles=None, rule="delaygo"):
    period_params = period_params or {}
    periods = ["fix1", "stim1", "delay1", "go1"]
    fps_by_period = {}
    angle_fps = {}
    for period in periods:
        p = period_params.get(period, {})
        central_seeds = fps_by_period.get("delay1", None) if period == "go1" else None
        fps, a_fps = search_period_fps(
            model, inputs, states, epochs, period,
            params=p,
            central_seeds=central_seeds,
            angles=angles,
            stim_angles=stim_angles,
            rule=rule,
        )
        fps_by_period[period] = fps
        angle_fps[period] = a_fps
    return fps_by_period, angle_fps


def save_pickle(obj, path):
    with open(path, "wb") as f:
        pickle.dump(obj, f)


def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)

### Projection helpers

Most figures project the high-dimensional hidden states into a low-dimensional basis.
The default basis is built from a PCA of the relevant period states; for panels that
need an output-readout axis, the third component is replaced by the `cos(response)`
readout weight and the basis is re-orthonormalized.

In [ ]:
def project_for_plot(z, D, b_cos):
    proj = z @ D
    proj[..., 2] += b_cos
    return proj


def project_for_plot_v2(z, D, b_cos=None):
    proj = z @ D
    if b_cos is not None:
        proj[..., 2] += b_cos
    return proj


def build_D_3d(states, epochs, w_cos):
    mem_states = get_period_states(states, epochs, "delay1").reshape(-1, states.shape[-1])
    pca = PCA(n_components=3).fit(mem_states)
    D = pca.components_.T
    D[:, 0] *= -1
    D[:, 1] *= -1
    D[:, 2] = np.asarray(w_cos).copy()
    return D, pca


def make_D_period(states, epochs, period, n_components=3):
    period_states = get_period_states(states, epochs, period).reshape(-1, states.shape[-1])
    pca = PCA(n_components=n_components).fit(period_states)
    D = pca.components_.T
    D[:, 0] *= -1
    D[:, 1] *= -1
    return D


def make_D_context_diff(states_pro, epochs_pro, states_anti, epochs_anti):
    s_f, e_f = get_epoch_slice(epochs_pro, "fix1", states_pro.shape[1])
    h_A = states_pro[0, s_f:e_f, :].T
    h_end = h_A[:, -1] - h_A[:, 0]
    h_B = states_anti[0, s_f:e_f, :].T
    h_diff = h_B[:, -1] - h_A[:, -1]
    D_fp = np.stack([h_diff, h_end], axis=1)
    D_fp_qr, _ = linalg.qr(D_fp, mode="economic")
    return D_fp_qr[:, 0]


def make_D_interp(states_pro, epochs_pro, states_anti, epochs_anti, w_cos, use_output_axis=False):
    stim_states = get_period_states(states_pro, epochs_pro, "stim1").reshape(-1, states_pro.shape[-1])
    pca = PCA(n_components=3).fit(stim_states)
    D = pca.components_.T
    D[:, 0] *= -1
    D[:, 1] *= -1
    if use_output_axis:
        D[:, 2] = np.asarray(w_cos).copy()
    else:
        D[:, 2] = make_D_context_diff(states_pro, epochs_pro, states_anti, epochs_anti)
    return D

### Task visualization helper

The function below plots a single noise-free example trial for any rule.  It is used
in the introductions of Parts I, II and III to show what the network has to do.

In [ ]:
def plot_example_trial(rule, seed=0):
    """Plot one example trial for `rule` (input channels, rule index, target output)."""
    inputs, targets, mask, conds = generate_trials(rule, n_trials=1, mode="random", seed=seed)
    inputs = inputs[0]
    targets = targets[0]
    t = np.arange(inputs.shape[0]) * DT
    epochs = conds[0]["epochs"]

    fig, axes = plt.subplots(5, 1, figsize=(6, 6), sharex=True)

    # Fixation input
    axes[0].plot(t, inputs[:, 0], color="black")
    axes[0].set_ylabel("Fixation input")
    axes[0].set_ylim(-0.1, 1.2)

    # Modality 1 (sin/cos)
    mod1 = inputs[:, 1:3]
    axes[1].plot(t, mod1[:, 0], label="sin", color="C0")
    axes[1].plot(t, mod1[:, 1], label="cos", color="C1")
    axes[1].set_ylabel("Modality 1")
    axes[1].legend(loc="upper right")

    # Modality 2 (sin/cos)
    mod2 = inputs[:, 3:5]
    axes[2].plot(t, mod2[:, 0], color="C0")
    axes[2].plot(t, mod2[:, 1], color="C1")
    axes[2].set_ylabel("Modality 2")

    # Rule one-hot
    rule_idx = int(np.argmax(inputs[0, RULE_START:]))
    axes[3].axhline(rule_idx, color="C2", linewidth=2)
    axes[3].set_ylabel("Rule index")
    axes[3].set_ylim(-0.5, 15.5)

    # Target output
    axes[4].plot(t, targets[:, 0], color="black", label="fixation target")
    axes[4].plot(t, targets[:, 1], color="C0", label="sin response")
    axes[4].plot(t, targets[:, 2], color="C1", label="cos response")
    axes[4].set_ylabel("Target output")
    axes[4].set_xlabel("Time (ms)")
    axes[4].legend(loc="upper left")

    for ax in axes:
        for period, (s, e) in epochs.items():
            if s is not None:
                ax.axvline(s * DT, color="gray", linestyle=":", linewidth=0.8)

    fig.suptitle(f"Example trial: {RULE_NAME.get(rule, rule)} ({rule})")
    plt.tight_layout()
    fig.savefig(os.path.join(FIG_DIR, f"example_trial_{rule}.png"), dpi=150, bbox_inches='tight')
    plt.show()

## Part I — Single-task MemoryPro network


We start with the simplest setting: a network trained on a single task, `delaygo` (MemoryPro).  The network must remember a stimulus direction through a delay and then respond in the same direction.  Figure 1 of the paper shows that this network implements a **ring attractor** that is reused between the memory period and the response period, rotating from output-null to output-potent space when fixation is released.

### 1.1 Task introduction: MemoryPro (`delaygo`)


The trial below shows the input structure.  The first input dimension is fixation (high until the response period).  Dimensions 1–4 encode two circular stimuli as `(sin θ, cos θ)` pairs.  The rule input (dimensions 5–19) is a one-hot vector that identifies the task.  The target output is fixation during fixation and the response direction during the go period.

In [ ]:
plot_example_trial('delaygo', seed=0)

### 1.2 Train or load the single-task network

The model is a 128-unit CTRNN with softplus activation.

In [ ]:
model_single = train_or_load(
    ["delaygo"], MODEL_DIR_SINGLE,
    latent_dim=LATENT_DIM, max_steps=6000)
model_single.eval()
print(model_single)

### 1.3 Generate test trials


We generate 400 test trials for `delaygo` and run the network deterministically. The resulting `states` tensor has shape `(n_trials, n_timesteps, latent_dim)`.

In [ ]:
RULE_SINGLE = "delaygo"
inputs_s, targets_s, mask_s, conds_s = generate_trials(
    RULE_SINGLE, n_trials=N_CONDITIONS, mode="test", seed=SEED)
outputs_s, states_s = run_forward(model_single, inputs_s)
epochs_s = conds_s[0]["epochs"]
T_s = inputs_s.shape[1]
print("Epochs:", epochs_s)
print("states shape:", states_s.shape, "inputs shape:", inputs_s.shape)

w_cos_s = model_single.readout_layer.weight.data[2].cpu().numpy()
b_cos_s = model_single.readout_layer.bias.data[2].cpu().numpy().item()
print(f"Readout cos: ||w||={np.linalg.norm(w_cos_s):.4f}, b={b_cos_s:.4f}")

### 1.4 variance explained by memory-period PCs


Before visualizing trajectories, we quantify how much variance of the memory-period activity is captured by its own principal components.  This gives the reader a reference for how faithful the 2-D/3-D projections are.

In [ ]:
def plot_fig1b(states, epochs, fig_dir):
    mem_states = get_period_states(states, epochs, "delay1").reshape(-1, states.shape[-1])
    pca_full = PCA(n_components=min(20, mem_states.shape[1])).fit(mem_states)
    var = pca_full.explained_variance_ratio_
    cumvar = np.cumsum(var)

    fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))
    axes[0].bar(np.arange(1, len(var) + 1), var, color="steelblue")
    axes[0].set_xlabel("PC")
    axes[0].set_ylabel("Variance explained")
    axes[0].set_title("Figure 1b — Variance per PC")
    axes[1].plot(np.arange(1, len(cumvar) + 1), cumvar, "o-", color="steelblue")
    axes[1].axhline(0.9, color="gray", linestyle="--", linewidth=0.8)
    axes[1].set_xlabel("Number of PCs")
    axes[1].set_ylabel("Cumulative variance")
    axes[1].set_title("Figure 1b — Cumulative variance")
    plt.tight_layout()
    out_path = os.path.join(fig_dir, "fig1_b.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    return var, cumvar


def compute_plot_limits(states, epochs, fps_by_period, D, b_cos, angles, stim_angles):
    """Compute consistent x/y/z limits across all panels."""
    all_proj = []
    for period in ["fix1", "stim1", "delay1", "go1"]:
        for theta in angles:
            ang_diff = np.abs(np.angle(np.exp(1j * (stim_angles - theta))))
            idx = int(np.argmin(ang_diff))
            ps = get_period_states(states[idx:idx + 1], epochs, period)[0]
            all_proj.append(project_for_plot(ps, D, b_cos))
        if fps_by_period[period]:
            fp_z = np.array([fp.z for fp in fps_by_period[period]])
            all_proj.append(project_for_plot(fp_z, D, b_cos))
    all_proj = np.concatenate(all_proj, axis=0)
    xlim = (all_proj[:, 0].min() - 1, all_proj[:, 0].max() + 1)
    ylim = (all_proj[:, 1].min() - 1, all_proj[:, 1].max() + 1)
    zlim = (all_proj[:, 2].min() - 0.1, all_proj[:, 2].max() + 0.1)
    return xlim, ylim, zlim


def _period_to_label(period):
    return {"fix1": "c", "stim1": "d", "delay1": "e", "go1": "f"}[period]


def plot_single_period_3d(period, states, epochs, fps_by_period, D, b_cos,
                          angles, stim_angles, xlim, ylim, zlim, fig_dir,
                          elev=20, azim=-60):
    """Plot one 3-D panel for Figure 1c-f and save it."""
    fig = plt.figure(figsize=(5.5, 5.5))
    ax = fig.add_subplot(111, projection="3d")

    for theta in angles:
        ang_diff = np.abs(np.angle(np.exp(1j * (stim_angles - theta))))
        idx = int(np.argmin(ang_diff))
        ps = get_period_states(states[idx:idx + 1], epochs, period)[0]
        proj = project_for_plot(ps, D, b_cos)
        color = plt.cm.hsv(theta / (2 * np.pi))
        lw = 2.5 if theta == 0 else 1.0
        ax.plot(proj[:, 0], proj[:, 1], proj[:, 2],
                color="red" if theta == 0 else color,
                alpha=0.85, linewidth=lw)
        ax.scatter(proj[0, 0], proj[0, 1], proj[0, 2],
                   color="red" if theta == 0 else color, marker="x", s=40)
        ax.scatter(proj[-1, 0], proj[-1, 1], proj[-1, 2],
                   color="red" if theta == 0 else color, marker="^", s=40)

    fp_points = np.array([fp.z for fp in fps_by_period[period]])
    if len(fp_points) > 0:
        fp_pc = project_for_plot(fp_points, D, b_cos)
        stable = np.array([fp.is_stable for fp in fps_by_period[period]])
        ax.scatter(fp_pc[stable, 0], fp_pc[stable, 1], fp_pc[stable, 2],
                   s=60, c="black", marker="o", zorder=5, label="stable FP")
        ax.scatter(fp_pc[~stable, 0], fp_pc[~stable, 1], fp_pc[~stable, 2],
                   s=60, c="red", marker="o", facecolors="none",
                   zorder=5, label="unstable/saddle FP")

    xx, yy = np.meshgrid(
        np.linspace(xlim[0], xlim[1], 2),
        np.linspace(ylim[0], ylim[1], 2),
    )
    zz = np.zeros_like(xx)
    ax.plot_surface(xx, yy, zz, alpha=0.08, color="gray")

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)
    ax.set_zlim(zlim)
    ax.set_xlabel("Memory PC1")
    ax.set_ylabel("Memory PC2")
    ax.set_zlabel("Output cos(theta)")
    ax.set_title(f"Figure 1{_period_to_label(period)} — {period}")
    ax.view_init(elev=elev, azim=azim)
    ax.legend(fontsize=7, loc="upper left")

    out_path = os.path.join(fig_dir, f"fig1_{_period_to_label(period)}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()
    return fig, ax


def find_ring_fps_for_alpha(model, u_alpha, states, epochs, params=None):
    """Find the full ring of fixed points under a single interpolated input."""
    s_d, e_d = get_epoch_slice(epochs, "delay1", states.shape[1])
    delay_end_states = states[:, e_d - 1, :]
    print(f"  alpha interpolation: single ring search from {len(delay_end_states)} delay-end states")
    return find_fps_for_period(model, u_alpha, delay_end_states, params=params)


def plot_slice(ax, alpha, ring_fps, period_for_traj, states, epochs, D, b_cos,
               stim_angles, angles, xlabel="Memory PC1", ylabel="Memory PC2",
               use_output_axis=False):
    """Plot a 2-D slice at a single alpha with trajectories and fixed points."""
    for theta in angles:
        ang_diff = np.abs(np.angle(np.exp(1j * (stim_angles - theta))))
        idx = int(np.argmin(ang_diff))
        ps = get_period_states(states[idx:idx + 1], epochs, period_for_traj)[0]
        proj = project_for_plot(ps, D, b_cos)
        color = plt.cm.hsv(theta / (2 * np.pi))
        lw = 2.5 if theta == 0 else 1.0
        x = proj[:, 0]
        y = proj[:, 2] if use_output_axis else proj[:, 1]
        ax.plot(x, y, color="red" if theta == 0 else color,
                alpha=0.85, linewidth=lw)
        ax.scatter(x[0], y[0], color="red" if theta == 0 else color,
                   marker="x", s=40)
        ax.scatter(x[-1], y[-1], color="red" if theta == 0 else color,
                   marker="^", s=40)

    if ring_fps:
        fp_z = np.array([fp.z for fp in ring_fps])
        fp_pc = project_for_plot(fp_z, D, b_cos)
        x = fp_pc[:, 0]
        y = fp_pc[:, 2] if use_output_axis else fp_pc[:, 1]
        stable = np.array([fp.is_stable for fp in ring_fps])
        ax.scatter(x[stable], y[stable], s=80, c="black", marker="o",
                   zorder=5, label="stable FP")
        ax.scatter(x[~stable], y[~stable], s=80, c="red", marker="o",
                   facecolors="none", zorder=5, label="unstable/saddle FP")

    ax.axhline(0, color="gray", linestyle="--", linewidth=0.5)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(f"alpha = {alpha:.2f}")
    ax.legend(fontsize=7, loc="upper right")


def plot_fig1g(interp_ring, alphas, states, epochs, D, b_cos,
               stim_angles, angles, fig_dir):
    fig = plt.figure(figsize=(13, 4))

    ax3d = fig.add_subplot(1, 3, 2, projection="3d")
    colors = plt.cm.plasma(np.linspace(0, 1, len(interp_ring)))
    for (alpha, ring_fps), c in zip(interp_ring, colors):
        if not ring_fps:
            continue
        fp_z = np.array([fp.z for fp in ring_fps])
        fp_pc = project_for_plot(fp_z, D, b_cos)
        ax3d.scatter([alpha] * len(ring_fps), fp_pc[:, 0], fp_pc[:, 1],
                     color=c, s=50, alpha=0.85)
    ax3d.set_xlabel("Fixation input alpha")
    ax3d.set_ylabel("Memory PC1")
    ax3d.set_zlabel("Memory PC2")
    ax3d.set_title("Figure 1g — delay1 -> go1 interpolation")
    ax3d.view_init(elev=20, azim=-60)

    ax0 = fig.add_subplot(1, 3, 1)
    plot_slice(ax0, 0.0, interp_ring[0][1], "delay1", states, epochs, D, b_cos,
               stim_angles, angles, xlabel="Memory PC1", ylabel="Memory PC2")

    ax1 = fig.add_subplot(1, 3, 3)
    plot_slice(ax1, 1.0, interp_ring[-1][1], "go1", states, epochs, D, b_cos,
               stim_angles, angles, xlabel="Memory PC1", ylabel="Memory PC2")

    sm = plt.cm.ScalarMappable(cmap="plasma", norm=plt.Normalize(0, 1))
    sm.set_array([])
    fig.colorbar(sm, ax=[ax3d, ax0, ax1], shrink=0.6, label="alpha (response)")

    fig.suptitle("Figure 1g — Fixed-point interpolation (PC1/PC2)", fontsize=12)
    plt.tight_layout()
    out_path = os.path.join(fig_dir, "fig1_g.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()


def plot_fig1h(interp_ring, alphas, states, epochs, D, b_cos,
               stim_angles, angles, fig_dir):
    fig = plt.figure(figsize=(4, 4))

    ax3d = fig.add_subplot(1, 1, 1, projection="3d")
    colors = plt.cm.plasma(np.linspace(0, 1, len(interp_ring)))
    for (alpha, ring_fps), c in zip(interp_ring, colors):
        if not ring_fps:
            continue
        fp_z = np.array([fp.z for fp in ring_fps])
        fp_pc = project_for_plot(fp_z, D, b_cos)
        ax3d.scatter(fp_pc[:, 0], fp_pc[:, 1], fp_pc[:, 2],
                     color=c, s=5, alpha=0.85)

    ax3d.set_xlabel("Fixation input alpha")
    ax3d.set_ylabel("Memory PC1")
    ax3d.set_zlabel("Output cos(theta)")
    ax3d.set_title("Figure 1h — delay1 -> go1 interpolation")
    ax3d.view_init(elev=20, azim=-60)

    fig.suptitle("Figure 1h — Fixed-point interpolation (PC1 / output cos)", fontsize=12)
    plt.tight_layout()
    out_path = os.path.join(fig_dir, "fig1_h.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()

In [ ]:
D_s, pca_s = build_D_3d(states_s, epochs_s, w_cos_s)
stim_angles_s = get_stim_angles(inputs_s, epochs_s)
angles_s = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
print("Memory PC explained variance:", pca_s.explained_variance_ratio_[:3])

plot_fig1b(states_s, epochs_s, FIG_DIR)

### 1.5 Search fixed points for each task period


Fixed points provide a "skeleton" of the dynamics during each task period.  We search for fixed points under the input corresponding to the onset of `fix1`, `stim1`, `delay1`, and `go1`.  Results are cached as `.pkl` files next to the model checkpoint.

In [ ]:
# Part I cache paths match the existing files in models/12/flexible_delaygo_128/.
def fp_cache_path(model_dir, prefix, rule=None, period_params=None,
                  n_angles=None, seed=None):
    return os.path.join(model_dir, f"fps_{prefix}.pkl")

In [ ]:
FP_PARAMS_SINGLE = {
    "fix1": {},
    "stim1": {"n_candidates": 200},
    "delay1": {},
    "go1": {},
}

cache_path_s = fp_cache_path(
    MODEL_DIR_SINGLE, "fig1", RULE_SINGLE,
    FP_PARAMS_SINGLE, N_ANGLES, SEED)

if os.path.exists(cache_path_s):
    fps_by_period_s = load_pickle(cache_path_s)
else:
    print("Computing fixed points for Figure 1c-f ...")
    fps_by_period_s, _ = compute_all_period_fps(
        model_single, inputs_s, states_s, epochs_s,
        period_params=FP_PARAMS_SINGLE,
        angles=angles_s,
        stim_angles=stim_angles_s,
        rule=RULE_SINGLE)
    save_pickle(fps_by_period_s, cache_path_s)

for period, fps in fps_by_period_s.items():
    print(f"  {period}: {len(fps)} fixed points")

### 1.6 trajectories and fixed points in each period


The next four cells plot the network trajectories and fixed points during the context, stimulus, memory, and response periods.  The projection uses memory-period PC1/PC2 and the `cos(response)` readout weight as the third axis.  The trajectory for stimulus angle `θ = 0` is highlighted in red.

In [ ]:
# Shared plot limits: To make the four period panels comparable, 
# we compute consistent x/y/z limits from all trajectories and fixed points.
xlim_s, ylim_s, zlim_s = compute_plot_limits(
    states_s, epochs_s, fps_by_period_s, D_s, b_cos_s, angles_s, stim_angles_s)
print(f"Plot limits: x={xlim_s}, y={ylim_s}, z={zlim_s}")

`fix1` period

In [ ]:
plot_single_period_3d(
    "fix1", states_s, epochs_s, fps_by_period_s, D_s, b_cos_s,
    angles_s, stim_angles_s, xlim_s, ylim_s, zlim_s, FIG_DIR,
    elev=20, azim=-60)

`stim1` period

In [ ]:
plot_single_period_3d(
    "stim1", states_s, epochs_s, fps_by_period_s, D_s, b_cos_s,
    angles_s, stim_angles_s, xlim_s, ylim_s, zlim_s, FIG_DIR,
    elev=20, azim=-60)

`delay1` period

In [ ]:
plot_single_period_3d(
    "delay1", states_s, epochs_s, fps_by_period_s, D_s, b_cos_s,
    angles_s, stim_angles_s, xlim_s, ylim_s, zlim_s, FIG_DIR,
    elev=20, azim=-60)

`go1` period

In [ ]:
plot_single_period_3d(
    "go1", states_s, epochs_s, fps_by_period_s, D_s, b_cos_s,
    angles_s, stim_angles_s, xlim_s, ylim_s, zlim_s, FIG_DIR,
    elev=20, azim=-60)

### 1.7 interpolation from memory to response input


We linearly interpolate the input from the memory-period setting (`fixation=1`, no stimulus) to the response-period setting (`fixation=0`, no stimulus).  At each intermediate input we search for the ring of fixed points.  A smooth transition indicates that the memory and response rings are the same dynamical motif.

In [ ]:
N_INTERP_SINGLE = 21
u_mem_s = period_input_with_rule(
    get_period_input_at_onset(inputs_s, epochs_s, "delay1"), RULE_SINGLE)
u_resp_s = period_input_with_rule(
    get_period_input_at_onset(inputs_s, epochs_s, "go1"), RULE_SINGLE)

interp_cache_s = fp_cache_path(
    MODEL_DIR_SINGLE, "fig1_interp", RULE_SINGLE,
    {"INTERP_FP_PARAMS": str({})}, N_INTERP_SINGLE, SEED)

if os.path.exists(interp_cache_s):
    interp_ring_s = load_pickle(interp_cache_s)
else:
    print("Interpolating fixed points ...")
    interp_ring_s = []
    alphas_s = np.linspace(0, 1, N_INTERP_SINGLE)
    for alpha in alphas_s:
        u = (1 - alpha) * u_mem_s + alpha * u_resp_s
        ring_fps = find_ring_fps_for_alpha(model_single, u, states_s, epochs_s)
        interp_ring_s.append((alpha, ring_fps))
        print(f"  alpha={alpha:.2f}: {len(ring_fps)} ring points")
    save_pickle(interp_ring_s, interp_cache_s)

alphas_s = np.linspace(0, 1, N_INTERP_SINGLE)
plot_fig1g(interp_ring_s, alphas_s, states_s, epochs_s, D_s, b_cos_s,
           stim_angles_s, angles_s, FIG_DIR)

same interpolation viewed in output-potent space: The same fixed-point ring is now projected onto memory PC1 and the `cos(response)` output axis.  The ring rotates from output-null to output-potent space as fixation is released.

In [ ]:
plot_fig1h(interp_ring_s, alphas_s, states_s, epochs_s, D_s, b_cos_s,
           stim_angles_s, angles_s, FIG_DIR)

## Part II — Two-task MemoryPro + MemoryAnti network


Next we train a network on two interleaved tasks: `delaygo` (MemoryPro) and `delayanti` (MemoryAnti).  The tasks receive the same stimulus but require opposite responses.  Figure 2 of the paper shows that the network flexibly switches between tasks by small changes in fixed-point locations, while sharing the underlying ring attractor.

### 2.1 Task introduction: MemoryAnti (`delayanti`)

MemoryAnti is identical to MemoryPro during the stimulus period, but the target response is shifted by π (opposite direction).  The network must therefore use the same rule input to read the same stimulus but produce the opposite motor output.

In [ ]:
plot_example_trial('delayanti', seed=0)

### 2.2 Train or load the two-task network

The two-task network has the same architecture but is trained on interleaved batches of `delaygo` and `delayanti` for 15,000 steps.

In [ ]:
RULE_PRO = "delaygo"
RULE_ANTI = "delayanti"

model_dual = train_or_load(
    [RULE_PRO, RULE_ANTI], MODEL_DIR_DUAL,
    latent_dim=LATENT_DIM, max_steps=15000)
model_dual.eval()
print(model_dual)

### 2.3 variance explained across Pro/Anti periods

Before the rule-input bifurcation diagrams, we build the projection bases and compute the variance matrix of Figure 2a.  The helper functions are defined first, then the actual plots are produced in the next two cells.

In [ ]:
PERIOD_NAMES_FIG2 = {"fix1": "Context", "stim1": "Stim", "delay1": "Memory", "go1": "Response"}
N_INTERP_FIG2 = 20
CMAP_DISCRETE = ["k", "w"]
MODEL_DIR = MODEL_DIR_DUAL

In [ ]:
def _fp_cache_key(rule_a, rule_b, period, params, n_interp, seed):
    h = hashlib.sha1()
    h.update(rule_a.encode())
    h.update(rule_b.encode())
    h.update(period.encode())
    h.update(str(sorted(params.items())).encode())
    h.update(str(n_interp).encode())
    h.update(str(seed).encode())
    return h.hexdigest()[:16]


def fp_interp_cache_path(model_dir, rule_a, rule_b, period, params, n_interp, seed):
    key = _fp_cache_key(rule_a, rule_b, period, params, n_interp, seed)
    return os.path.join(model_dir, f"fps_fig2_{period}_{key}.pkl")

In [ ]:
def collect_period_states(model, rule, period, n_trials=N_CONDITIONS):
    inputs, targets, mask, conds = generate_trials(
        rule, n_trials=n_trials, mode="test", seed=SEED
    )
    outputs, states = run_forward(model, inputs)
    epochs = conds[0]["epochs"]
    return get_period_states(states, epochs, period).reshape(-1, states.shape[-1])


def compute_variance_matrix_fig2a(model, n_components=80):
    labels = []
    state_sets = []
    for rule, ctx_name in [(RULE_PRO, "Pro"), (RULE_ANTI, "Anti")]:
        for period in ["fix1", "stim1", "delay1", "go1"]:
            labels.append(f"{ctx_name} {PERIOD_NAMES_FIG2[period]}")
            state_sets.append(collect_period_states(model, rule, period))

    n_sets = len(state_sets)
    var_mat = np.zeros((n_sets, n_sets, n_components))
    for i, fit_states in enumerate(state_sets):
        n_comp = min(n_components, fit_states.shape[1], fit_states.shape[0])
        pca = PCA(n_components=n_comp).fit(fit_states)
        for j, test_states in enumerate(state_sets):
            test_centered = test_states - pca.mean_
            proj = test_centered @ pca.components_.T
            var_explained = np.var(proj, axis=0)
            total_var = np.var(test_centered, axis=0).sum()
            ratios = var_explained / (total_var + 1e-12)
            var_mat[i, j, :n_comp] = ratios
    return var_mat, labels


def plot_fig2a_left(var_mat, labels, fig_dir, n_dims=2):
    summed = np.sum(var_mat[:, :, :n_dims], axis=2)
    fig, ax = plt.subplots(figsize=(5.5, 4.5), tight_layout=True, facecolor="white")
    im = ax.imshow(summed, cmap="cividis", vmin=0, vmax=1)
    ax.set_xlabel("Fit", fontsize=14)
    ax.set_ylabel("Test", fontsize=14)
    ax.set_title(f"Fraction var. in first {n_dims} dims", fontsize=13)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(labels, fontsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["bottom"].set_visible(False)
    ax.spines["left"].set_visible(False)
    c = plt.colorbar(im, ax=ax, shrink=0.8)
    c.outline.set_visible(False)
    out_path = os.path.join(fig_dir, "fig2_a_left.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()


def plot_fig2a_right(model, fig_dir, n_components=15):
    pro_states = collect_period_states(model, RULE_PRO, "delay1")
    anti_states = collect_period_states(model, RULE_ANTI, "delay1")

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6.5, 3), tight_layout=True, facecolor="white")
    colors = ["k", "orange"]

    for ax, states, title, color in [(ax1, pro_states, "MemoryPro", colors[0]),
                                      (ax2, anti_states, "MemoryAnti", colors[1])]:
        n_comp = min(n_components, states.shape[1], states.shape[0])
        pca = PCA(n_components=n_comp).fit(states)
        cumvar = np.cumsum(pca.explained_variance_ratio_)
        ax.plot(np.arange(1, len(cumvar) + 1), cumvar, ".-",
                c=color, alpha=0.6, linewidth=2, markersize=10)
        ax.set_title(title, fontsize=14)
        ax.set_xlabel("N PCs", fontsize=12)
        ax.set_xlim([-0.5, len(cumvar) + 0.5])
        ax.set_ylim([-0.05, 1.05])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.tick_params(axis="both", labelsize=10)

    ax1.set_ylabel("Var Expl.", fontsize=12)
    ax2.set_yticks([])
    ax2.spines["left"].set_visible(False)
    out_path = os.path.join(fig_dir, "fig2_a_right.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()


def make_rule_interp_input(base_input, rule_a, rule_b, alpha):
    v_a = period_input_with_rule(base_input.copy(), rule_a)
    v_b = period_input_with_rule(base_input.copy(), rule_b)
    return (1 - alpha) * v_a + alpha * v_b


def get_out_angles(targets, epochs):
    last = targets[:, -1, 1:3]
    return np.mod(np.arctan2(last[:, 0], last[:, 1]), 2 * np.pi)


def same_mov_inds_py3(trial_master, trial_temp):
    n_trials = trial_master["targets"].shape[0]
    out_master = trial_master["out_angles"]
    out_temp = trial_temp["out_angles"]

    indices = np.zeros(n_trials, dtype=int)
    for ii in range(n_trials):
        loc_diff = np.abs(np.mod(out_temp - out_master[ii] + np.pi, 2 * np.pi) - np.pi)
        align_ind = int(np.argmin(loc_diff))
        indices[ii] = align_ind

    trial_temp_new = {k: v[indices] for k, v in trial_temp.items()}
    return trial_temp_new


def seed_states_for_period(states, epochs, period):
    T = states.shape[1]
    if period == "fix1":
        s_prev, e_prev = get_epoch_slice(epochs, "delay1", T)
    elif period == "stim1":
        s_prev, e_prev = get_epoch_slice(epochs, "fix1", T)
    elif period == "delay1":
        s_prev, e_prev = get_epoch_slice(epochs, "stim1", T)
    elif period == "go1":
        s_prev, e_prev = get_epoch_slice(epochs, "delay1", T)
    else:
        s_prev, e_prev = 0, T
    return states[:, e_prev - 1, :]


def compute_rule_interp_fps(model, inputs_pro, states_pro, epochs_pro,
                            period, n_interp=N_INTERP_FIG2, params=None):
    params = params or {}
    seed = seed_states_for_period(states_pro, epochs_pro, period)
    u_base = period_input_with_rule(
        get_period_input_at_onset(inputs_pro, epochs_pro, period), RULE_PRO)

    alphas = np.linspace(0, 1, n_interp)
    fp_by_alpha = []
    for alpha in alphas:
        u = make_rule_interp_input(u_base, RULE_PRO, RULE_ANTI, alpha)
        u_non_rule = period_input_with_rule(
            get_period_input_at_onset(inputs_pro, epochs_pro, period), RULE_PRO)
        u[:RULE_START] = u_non_rule[:RULE_START]
        u[RULE_START + len(RULES_ALL):] = u_non_rule[RULE_START + len(RULES_ALL):]

        fps = find_fps_for_period(model, u, seed, params=params)
        fp_by_alpha.append((alpha, dedup_fps(fps, tol=params.get("dedup_tol", 1e-2))))
    return alphas, fp_by_alpha


def period_to_fig2_label(period, interp3d=True):
    if interp3d:
        return {"stim1": "c", "fix1": "e", "delay1": "g", "go1": "i"}[period]
    return {"fix1": "b", "stim1": "d", "delay1": "f", "go1": "h"}[period]


def _stim_color(theta):
    return plt.cm.hsv(np.mod(theta / (2 * np.pi), 1.0))


def plot_highlighted_trajectory_2d(ax, traj_pc, color, fill_color,
                                   lw=6, alpha=1.0, markers=True):
    ax.plot(traj_pc[:, 0], traj_pc[:, 1], c=color, linewidth=lw, alpha=alpha)
    ax.plot(traj_pc[:, 0], traj_pc[:, 1], c=fill_color, linewidth=lw / 2, alpha=alpha)
    if markers:
        ax.scatter(traj_pc[0, 0], traj_pc[0, 1], s=200, marker="x",
                   facecolors=color, edgecolors=fill_color, linewidths=1.5, alpha=alpha)
        ax.scatter(traj_pc[-1, 0], traj_pc[-1, 1], s=300, marker="^",
                   facecolors=fill_color, edgecolors=color, linewidths=1.5, alpha=alpha)


def plot_highlighted_trajectory_3d(ax, traj_pc, color, fill_color,
                                   lw=6, alpha=1.0, markers=True):
    ax.plot(traj_pc[:, 0], traj_pc[:, 1], traj_pc[:, 2],
            c=color, linewidth=lw, alpha=alpha)
    ax.plot(traj_pc[:, 0], traj_pc[:, 1], traj_pc[:, 2],
            c=fill_color, linewidth=lw / 2, alpha=alpha)
    if markers:
        ax.scatter(traj_pc[0, 0], traj_pc[0, 1], traj_pc[0, 2], s=200, marker="x",
                   c=color, linewidth=1.5, alpha=alpha)
        ax.scatter(traj_pc[-1, 0], traj_pc[-1, 1], traj_pc[-1, 2], s=300, marker="^",
                   facecolors=fill_color, edgecolors=color, linewidths=1.5, alpha=alpha)


def plot_task_dynamics_panel(ax, period, states, epochs, stim_angles,
                             D, b_cos, theta_highlight, rule_idx,
                             n_background=8, lw=6):
    fill_color = CMAP_DISCRETE[rule_idx]

    bg_inds = np.linspace(0, len(states) - 1, n_background, dtype=int)
    for idx in bg_inds:
        ps = get_period_states(states[idx:idx + 1], epochs, period)[0]
        proj = project_for_plot_v2(ps, D, b_cos)[:, :2]
        c = _stim_color(stim_angles[idx])
        ax.plot(proj[:, 0], proj[:, 1], c=c, linewidth=lw, alpha=0.1)

    ang_diff = np.abs(np.angle(np.exp(1j * (stim_angles - theta_highlight))))
    idx = int(np.argmin(ang_diff))
    ps = get_period_states(states[idx:idx + 1], epochs, period)[0]
    proj = project_for_plot_v2(ps, D, b_cos)[:, :2]
    c = _stim_color(stim_angles[idx])
    plot_highlighted_trajectory_2d(ax, proj, c, fill_color, lw=lw, alpha=0.9)

    ax.set_xlabel("State PC1")
    ax.set_ylabel("State PC2")
    ax.set_title("MemoryPro" if rule_idx == 0 else "MemoryAnti")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def plot_bifurcation_panel(ax, period, alphas, fp_by_alpha, D, b_cos,
                           three_d=True, pc_axis=0):
    colors = plt.cm.plasma(np.linspace(0, 1, len(fp_by_alpha)))

    for (alpha, fps), c in zip(fp_by_alpha, colors):
        if not fps:
            continue
        fp_z = np.array([fp.z for fp in fps])
        fp_pc = project_for_plot_v2(fp_z, D, b_cos)
        stable = np.array([fp.is_stable for fp in fps])

        if three_d:
            ax.scatter(np.full(stable.sum(), alpha), fp_pc[stable, 0], fp_pc[stable, 1],
                       s=40, c=[c], marker="o", alpha=0.7, zorder=5)
            ax.scatter(np.full((~stable).sum(), alpha), fp_pc[~stable, 0], fp_pc[~stable, 1],
                       s=40, c=[c], marker="o", facecolors="none", alpha=0.7, zorder=5)
        else:
            ax.scatter([alpha] * stable.sum(), fp_pc[stable, pc_axis],
                       s=60, c=[c], marker="o", alpha=0.7, zorder=5)
            ax.scatter([alpha] * (~stable).sum(), fp_pc[~stable, pc_axis],
                       s=60, c=[c], marker="o", facecolors="none", alpha=0.7, zorder=5)

    if three_d:
        ax.set_xlabel(r"Rule Input $\alpha$")
        ax.set_ylabel("State PC1")
        ax.set_zlabel("State PC2")
    else:
        ax.set_xlabel(r"Rule Input $\alpha$")
        ax.set_ylabel(f"State PC{pc_axis + 1}")
        ax.set_xlim([-0.05, 1.05])
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)


def plot_fig2_bdfh(period, model, inputs_pro, states_pro, epochs_pro,
                   states_anti, epochs_anti, D, b_cos, angles,
                   stim_angles_pro, stim_angles_anti, fig_dir,
                   three_d=True, pc_axis=0, params=None):
    params = params or {}
    print(f"\nComputing rule interpolation for {period} ...")

    cache_path = fp_interp_cache_path(
        MODEL_DIR, RULE_PRO, RULE_ANTI, period, params, N_INTERP_FIG2, SEED)
    if os.path.exists(cache_path):
        alphas, fp_by_alpha = load_pickle(cache_path)
    else:
        alphas, fp_by_alpha = compute_rule_interp_fps(
            model, inputs_pro, states_pro, epochs_pro,
            period, n_interp=N_INTERP_FIG2, params=params
        )
        save_pickle((alphas, fp_by_alpha), cache_path)

    print(f"  {period}: alpha=0..1, fixed points per alpha: " +
          ", ".join(str(len(fps)) for _, fps in fp_by_alpha))

    fig = plt.figure(figsize=(16, 5), tight_layout=True, facecolor="white")

    ax_left = fig.add_subplot(1, 3, 1)
    theta_highlight = angles[0] if len(angles) > 0 else 0.0
    plot_task_dynamics_panel(ax_left, period, states_pro, epochs_pro,
                             stim_angles_pro, D, b_cos, theta_highlight,
                             rule_idx=0, n_background=8, lw=6)

    if three_d:
        ax_center = fig.add_subplot(1, 3, 2, projection="3d")
    else:
        ax_center = fig.add_subplot(1, 3, 2)
    plot_bifurcation_panel(ax_center, period, alphas, fp_by_alpha,
                           D, b_cos, three_d=three_d, pc_axis=pc_axis)
    ax_center.set_title(f"{period} rule interpolation")

    ax_right = fig.add_subplot(1, 3, 3)
    plot_task_dynamics_panel(ax_right, period, states_anti, epochs_anti,
                             stim_angles_anti, D, b_cos, theta_highlight,
                             rule_idx=1, n_background=8, lw=6)

    label = period_to_fig2_label(period, interp3d=False)
    fig.suptitle(f"Figure 2{label} — {period} bifurcation", fontsize=14)

    out_path = os.path.join(fig_dir, f"fig2_{label}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()
    return fig, (ax_left, ax_center, ax_right)


def plot_fig2_acegi(period, states_pro, epochs_pro, states_anti, epochs_anti,
                    D, b_cos, angles, stim_angles_pro, stim_angles_anti,
                    fig_dir, use_output_axis=False, elev=20, azim=-60):
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection="3d")

    for rule_idx, (states, epochs, stim_angles) in enumerate(
        [(states_pro, epochs_pro, stim_angles_pro),
         (states_anti, epochs_anti, stim_angles_anti)]
    ):
        fill_color = CMAP_DISCRETE[rule_idx]

        bg_inds = np.linspace(0, len(states) - 1, 8, dtype=int)
        for idx in bg_inds:
            ps = get_period_states(states[idx:idx + 1], epochs, period)[0]
            proj = project_for_plot_v2(ps, D, b_cos)
            c = _stim_color(stim_angles[idx])
            ax.plot(proj[:, 0], proj[:, 1], proj[:, 2],
                    c=c, linewidth=4, alpha=0.15)

        for theta in angles:
            ang_diff = np.abs(np.angle(np.exp(1j * (stim_angles - theta))))
            idx = int(np.argmin(ang_diff))
            ps = get_period_states(states[idx:idx + 1], epochs, period)[0]
            proj = project_for_plot_v2(ps, D, b_cos)
            c = _stim_color(stim_angles[idx])
            lw = 5.0 if theta == 0 else 2.5
            plot_highlighted_trajectory_3d(ax, proj, c, fill_color,
                                           lw=lw, alpha=0.85)

    ax.set_xlabel("State PC1")
    ax.set_ylabel("State PC2")
    ax.set_zlabel("Output cos(theta)" if use_output_axis else "Context endpt. diff")
    label = period_to_fig2_label(period, interp3d=True)
    ax.set_title(f"Figure 2{label} — {period} dynamics")
    ax.view_init(elev=elev, azim=azim)

    out_path = os.path.join(fig_dir, f"fig2_{label}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()
    return fig, ax

### 2.4 Generate and align Pro/Anti trials

We generate test trials for both tasks.  For visualization, Anti trials are reordered so that their output directions match the Pro trials.  This makes it easy to compare how the same stimulus leads to opposite responses.

In [ ]:
w_cos_d = model_dual.readout_layer.weight.data[2].cpu().numpy()
b_cos_d = model_dual.readout_layer.bias.data[2].cpu().numpy().item()

inputs_pro, targets_pro, _, conds_pro = generate_trials(
    RULE_PRO, n_trials=N_CONDITIONS, mode="test", seed=SEED)
outputs_pro, states_pro = run_forward(model_dual, inputs_pro)
epochs_pro = conds_pro[0]["epochs"]

inputs_anti, targets_anti, _, conds_anti = generate_trials(
    RULE_ANTI, n_trials=N_CONDITIONS, mode="test", seed=SEED)
outputs_anti, states_anti = run_forward(model_dual, inputs_anti)
epochs_anti = conds_anti[0]["epochs"]

stim_angles_pro = get_stim_angles(inputs_pro, epochs_pro)
stim_angles_anti = get_stim_angles(inputs_anti, epochs_anti)

master_pro = {
    "inputs": inputs_pro, "targets": targets_pro,
    "stim_angles": stim_angles_pro,
    "out_angles": get_out_angles(targets_pro, epochs_pro),
}
temp_anti = {
    "inputs": inputs_anti, "targets": targets_anti,
    "stim_angles": stim_angles_anti,
    "out_angles": get_out_angles(targets_anti, epochs_anti),
}
aligned_anti = same_mov_inds_py3(master_pro, temp_anti)
inputs_anti = aligned_anti["inputs"]
targets_anti = aligned_anti["targets"]
stim_angles_anti = aligned_anti["stim_angles"]
_, states_anti = run_forward(model_dual, inputs_anti)

angles_d = np.linspace(0, 2 * np.pi, N_ANGLES, endpoint=False)
print("Epochs pro:", epochs_pro)
print("Epochs anti:", epochs_anti)
print(f"Aligned {len(stim_angles_anti)} Anti trials to Pro output directions.")

### 2.4 Build projection bases

Figure 2 uses three kinds of bases:
* **Per-period bases** (`D_period`) for the bifurcation diagrams (Fig 2b/d/f/h).
* **Context-difference basis** (`D_context`) for panels where the third axis should
  separate Pro and Anti context endpoints (Fig 2c/e).
* **Output-readout basis** (`D_output`) for panels where the third axis is the motor
  output (Fig 2g/i).

In [ ]:
D_context_d = make_D_interp(
    states_pro, epochs_pro, states_anti, epochs_anti,
    w_cos_d, use_output_axis=False)
D_output_d = make_D_interp(
    states_pro, epochs_pro, states_anti, epochs_anti,
    w_cos_d, use_output_axis=True)

D_period_d = {
    "fix1": make_D_period(states_pro, epochs_pro, "fix1"),
    "stim1": make_D_period(states_pro, epochs_pro, "stim1"),
    "delay1": make_D_period(states_pro, epochs_pro, "delay1"),
    "go1": make_D_period(states_pro, epochs_pro, "go1"),
}
print("Built D_context, D_output, and per-period bifurcation bases.")

### 2.5 Figure 2a — variance explained across Pro/Anti periods

Figure 2a quantifies how much variance of one task period is explained by the top PCs of another task period.  Shared dynamical motifs should produce high cross-period variance explained.

In [ ]:
FP_PARAMS_FIG2 = {
    "fix1":   {"noise_scale": 0.01, "q_thresh": 5e-2, "dedup_tol": 8e-2},
    "stim1":  {"n_candidates": 200, "noise_scale": 0.01, "q_thresh": 5e-2, "dedup_tol": 8e-2},
    "delay1": {"noise_scale": 0.01, "q_thresh": 5e-2, "dedup_tol": 8e-2},
    "go1":    {"noise_scale": 0.01, "q_thresh": 5e-2, "dedup_tol": 8e-2},
}

var_mat_d, labels_d = compute_variance_matrix_fig2a(model_dual, n_components=80)
plot_fig2a_left(var_mat_d, labels_d, FIG_DIR, n_dims=2)

In [ ]:
plot_fig2a_right(model_dual, FIG_DIR, n_components=15)

### 2.6 Figures 2b–i — rule-input bifurcation diagrams


For each of the four task periods we linearly interpolate the rule input from
MemoryPro (α=0) to MemoryAnti (α=1) and search for fixed points at each α.  The left
and right panels show the Pro and Anti dynamics for the same stimulus direction; the
middle panel shows the bifurcation diagram.  Trajectories for the highlighted
direction use a black fill for Pro and a white fill for Anti.

#### Figure 2b — `fix1` bifurcation diagram

In [ ]:
plot_fig2_bdfh(
    "fix1", model_dual, inputs_pro, states_pro, epochs_pro,
    states_anti, epochs_anti, D_period_d["fix1"], None, angles_d,
    stim_angles_pro, stim_angles_anti, FIG_DIR,
    three_d=True, pc_axis=0, params=FP_PARAMS_FIG2["fix1"])

#### Figure 2c — `fix1` dynamics

In [ ]:
plot_fig2_acegi(
    "fix1", states_pro, epochs_pro, states_anti, epochs_anti,
    D_context_d, None, angles_d, stim_angles_pro, stim_angles_anti,
    FIG_DIR, use_output_axis=False, elev=20, azim=-60)

#### Figure 2d — `stim1` bifurcation diagram

In [ ]:
plot_fig2_bdfh(
    "stim1", model_dual, inputs_pro, states_pro, epochs_pro,
    states_anti, epochs_anti, D_period_d["stim1"], None, angles_d,
    stim_angles_pro, stim_angles_anti, FIG_DIR,
    three_d=False, pc_axis=0, params=FP_PARAMS_FIG2["stim1"])

#### Figure 2e — `stim1` dynamics

In [ ]:
plot_fig2_acegi(
    "stim1", states_pro, epochs_pro, states_anti, epochs_anti,
    D_context_d, None, angles_d, stim_angles_pro, stim_angles_anti,
    FIG_DIR, use_output_axis=False, elev=20, azim=-60)

#### Figure 2f — `delay1` bifurcation diagram

In [ ]:
plot_fig2_bdfh(
    "delay1", model_dual, inputs_pro, states_pro, epochs_pro,
    states_anti, epochs_anti, D_period_d["delay1"], b_cos_d, angles_d,
    stim_angles_pro, stim_angles_anti, FIG_DIR,
    three_d=True, pc_axis=0, params=FP_PARAMS_FIG2["delay1"])

#### Figure 2g — `delay1` dynamics

In [ ]:
plot_fig2_acegi(
    "delay1", states_pro, epochs_pro, states_anti, epochs_anti,
    D_output_d, b_cos_d, angles_d, stim_angles_pro, stim_angles_anti,
    FIG_DIR, use_output_axis=True, elev=20, azim=-60)

#### Figure 2h — `go1` bifurcation diagram

In [ ]:
plot_fig2_bdfh(
    "go1", model_dual, inputs_pro, states_pro, epochs_pro,
    states_anti, epochs_anti, D_period_d["go1"], b_cos_d, angles_d,
    stim_angles_pro, stim_angles_anti, FIG_DIR,
    three_d=True, pc_axis=0, params=FP_PARAMS_FIG2["go1"])

#### Figure 2i — `go1` dynamics

In [ ]:
plot_fig2_acegi(
    "go1", states_pro, epochs_pro, states_anti, epochs_anti,
    D_output_d, b_cos_d, angles_d, stim_angles_pro, stim_angles_anti,
    FIG_DIR, use_output_axis=True, elev=20, azim=-60)
plt.show()

## Part III — 15-task flexible multitask network

Finally we turn to the full 15-task network.  Here the paper asks two questions:

1. Are dynamical motifs organized into clusters of units?  **Figure 3A** shows the
   variance-matrix atlas obtained by clustering units according to their variance
   across task periods.
2. Do similar tasks share attractors?  **Figures 3D/E** interpolate between task pairs
   that the variance matrix predicts to share motifs.

**Figure 4** then examines the geometry of task representations: tasks with similar
stimulus computations start from nearby initial conditions and evolve in similar
directions.

### 3.1 Task introduction: example tasks for Figures 3 and 4

The 15-task set contains delayed-response, memory, reaction-timed, decision-making,
and delay-match tasks.  For the fixed-point analyses in Part III we focus on four
example tasks:

* `dmcgo` / `dmcnogo` — category-memory tasks (shared point attractors, Fig 3D).
* `delaydm1` / `delaydm2` — continuous-memory tasks (shared ring attractor, Fig 3E).

The next four cells show one example trial for each of these tasks.

#### Example trial: `dmcgo`

In [ ]:
plot_example_trial('dmcgo', seed=0)

#### Example trial: `dmcnogo`

In [ ]:
plot_example_trial('dmcnogo', seed=0)

#### Example trial: `delaydm1`

In [ ]:
plot_example_trial('delaydm1', seed=0)

#### Example trial: `delaydm2`

In [ ]:
plot_example_trial('delaydm2', seed=0)

### 3.2 Train or load the 15-task network

The full network is trained on all 15 tasks with context-dependent delay DM tasks oversampled 10×, following the paper.  Training takes up to 30,000 steps.

In [ ]:
model_multi = train_or_load(
    RULES_ALL, MODEL_DIR_MULTI,
    latent_dim=LATENT_DIM, max_steps=30000)
model_multi.eval()
print(model_multi)

### 3.3 Figure 3A — dynamic-modules atlas

We compute, for every unit and every task period, the variance of activity across
stimulus conditions.  After normalizing each unit by its maximum variance, we perform
hierarchical clustering on both units and task periods.  Block structure in the sorted
variance matrix reveals dynamical motifs: groups of units that are active during
groups of task periods with shared computations.

In [ ]:
# Constants used by the Figure 3/4 helper functions below.
N_RULE_FIG3 = 15
N_HIDDEN_FIG3 = 128
COLOR_PALETTE_FIG3 = ['#377eb8', '#ff7f00', '#4daf4a',
                      '#f781bf', '#a65628', '#984ea3',
                      '#e41a1c', '#dede00']

N_RULE_FIG4 = 15
N_HIDDEN_FIG4 = 128
RULE_START_FIG4 = 5
REACT_TASK_INDICES_FIG4 = [1, 4, 11, 12, 13, 14]
COLOR_SET_FIG4 = [
    'dodgerblue', 'limegreen', 'dodgerblue',
    'r', 'orange', 'r',
    'steelblue', 'steelblue', 'steelblue', 'steelblue', 'steelblue',
    'limegreen', 'orange', 'plum', 'plum',
]
RULE_SET_NAMES_FIG4 = [
    'DelayPro', 'ReactPro', 'MemoryPro',
    'DelayAnti', 'ReactAnti', 'MemoryAnti',
    'IntegrationModality1', 'IntegrationModality2',
    'CtxtIntModality1', 'CtxtIntModality2', 'IntegrationMultimodal',
    'ReactMatch2Sample', 'ReactNonMatch2Sample',
    'ReactCategoryPro', 'ReactCategoryAnti',
]
TASK_LABELS_FIG4 = {
    'all': list(range(15)),
    'Category Motif': [13, 14],
    'Different Motifs': [3, 1],
}

In [ ]:
# Part III functions reference the global MODEL_DIR variable.
MODEL_DIR = MODEL_DIR_MULTI

In [ ]:
def _fp_cache_key(rule_a, rule_b, period, params, n_interp, seed):
    h = hashlib.sha1()
    h.update(rule_a.encode())
    h.update(rule_b.encode())
    h.update(period.encode())
    h.update(str(sorted(params.items()) if isinstance(params, dict) else params).encode())
    h.update(str(n_interp).encode())
    h.update(str(seed).encode())
    return h.hexdigest()[:16]


def fp_interp_cache_path(model_dir, rule_a, rule_b, period, params, n_interp, seed):
    key = _fp_cache_key(rule_a, rule_b, period, params, n_interp, seed)
    return os.path.join(model_dir, f"fps_fig3_{period}_{key}.pkl")

In [ ]:
def decode_angle_from_ring(ring, pref=None):
    if pref is None:
        pref = np.arange(0, 2 * np.pi, 2 * np.pi / ring.shape[-1])
    ring = np.asarray(ring)
    s = ring.sum(axis=-1)
    s = np.where(s == 0, 1, s)
    sin = np.sum(ring * np.sin(pref), axis=-1) / s
    cos = np.sum(ring * np.cos(pref), axis=-1) / s
    return np.mod(np.arctan2(sin, cos), 2 * np.pi)


def decode_stim1_angle_fig3(inputs, epochs):
    s, e = get_epoch_slice(epochs, "stim1", inputs.shape[1])
    t = (s + e) // 2
    ring1 = inputs[:, t, 1:3]
    ring2 = inputs[:, t, 3:5]
    ang1 = np.mod(np.arctan2(ring1[:, 0], ring1[:, 1]), 2 * np.pi)
    ang2 = np.mod(np.arctan2(ring2[:, 0], ring2[:, 1]), 2 * np.pi)
    use2 = np.abs(ring1).sum(axis=-1) < 1e-3
    ang = np.where(use2, ang2, ang1)
    return ang


def align_trials_by_output(master_out, temp_out):
    master_out = np.asarray(master_out)
    temp_out = np.asarray(temp_out)
    master_valid = master_out >= 0
    temp_valid = temp_out >= 0

    indices = np.zeros(len(master_out), dtype=int)
    for i in range(len(master_out)):
        if master_valid[i]:
            valid = np.where(temp_valid)[0]
            diff = np.abs(np.mod(temp_out[valid] - master_out[i] + np.pi, 2 * np.pi) - np.pi)
            indices[i] = valid[int(np.argmin(diff))]
        else:
            invalid = np.where(~temp_valid)[0]
            if len(invalid):
                indices[i] = invalid[0]
            else:
                indices[i] = 0
    return indices


def compute_epoch_variance_fig3(model, rules=RULES_ALL):
    h_all_byepoch = {}

    for rule in rules:
        inputs, targets, mask, conds = generate_trials(rule, mode="test", sigma_x=0.0, seed=SEED)
        outputs, states = run_forward(model, inputs)
        epochs = conds[0]["epochs"]

        for e_name, (e_start, e_end) in epochs.items():
            if "fix" in e_name:
                continue
            if e_start is None:
                e_start = 0
            if e_end is None:
                e_end = states.shape[1]
            h_all_byepoch[(rule, e_name)] = states[:, e_start:e_end, :]

    keys = list(h_all_byepoch.keys())
    epoch_first = [k[1] for k in keys]
    ind_sort = np.argsort(epoch_first, kind="mergesort")
    keys = [keys[i] for i in ind_sort]

    h_var_all = np.zeros((states.shape[-1], len(keys)))
    for i, key in enumerate(keys):
        val = h_all_byepoch[key]
        h_var_all[:, i] = val.reshape(-1, states.shape[-1]).var(axis=0)

    return h_var_all, keys


def find_opt_clust_num_fig3(D, Y):
    n_clusters = range(3, min(len(Y), 40))
    scores = []
    for n_cluster in n_clusters:
        labels = fcluster(Y, n_cluster, criterion="maxclust")
        scores.append(silhouette_score(D, labels))
    return n_clusters[int(np.argmax(scores))]


def make_cluster_midpoint_labels_fig3(clust):
    d = np.concatenate(([-1], np.where(np.diff(clust))[0], [len(clust) - 1]))
    mid = np.zeros(len(d))
    cluster_size = np.zeros(len(d))
    for xi in range(len(d) - 1):
        cluster_size[xi] = d[xi + 1] - d[xi]
        mid[xi] = d[xi + 1] - cluster_size[xi] / 2 + 0.5
    return cluster_size, mid


def cluster_letter_label_fig3(i):
    i = int(i)
    label = ""
    while True:
        i, r = divmod(i - 1, 26)
        label = string.ascii_lowercase[r] + label
        if i == 0:
            return label


def plot_epoch_rects_fig3(ax, epoch_binary, e_set, n_cols,
                          rect_height=0.8, y_base=0.0):
    e_color = cm.get_cmap("terrain")
    for ei, e_name in enumerate(e_set):
        c = e_color(ei / len(e_set))
        for ind in np.where(epoch_binary[e_name])[0]:
            rect = mpatches.Rectangle(
                (ind - 0.5, y_base + ei * rect_height),
                1.0, rect_height,
                fill=True, color=c, facecolor=c, alpha=0.7, clip_on=False)
            ax.add_patch(rect)
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(0, len(e_set) * rect_height)
    ax.axis("off")


def plot_fig3a(model, fig_dir):
    print("\n=== Figure 3A: dynamic-modules atlas ===")
    h_var_all, keys = compute_epoch_variance_fig3(model)

    ind_active = np.where(h_var_all.sum(axis=1) > 1e-3)[0]
    h_var = h_var_all[ind_active, :]
    h_norm = (h_var.T / h_var.max(axis=1)).T

    D = h_norm

    max_d = 2.5
    method = "ward"
    criterion = "distance"
    Y_row = linkage(D, method=method)
    clusters_row = fcluster(Y_row, max_d, criterion=criterion)
    Z_row = dendrogram(Y_row, orientation="left", no_plot=True,
                       color_threshold=max_d, above_threshold_color="gray")

    index_left = Z_row["leaves"]
    D_sorted = D[index_left, :]
    clusters_row_sorted = clusters_row[index_left]

    Y_col = linkage(D_sorted.T, method=method)
    cel_opt = find_opt_clust_num_fig3(D_sorted.T, Y_col)
    clusters_col = fcluster(Y_col, cel_opt, criterion="maxclust")
    Z_col = dendrogram(Y_col, orientation="top", no_plot=True,
                       color_threshold=cel_opt, above_threshold_color="gray")
    index_top = Z_col["leaves"]
    D_sorted = D_sorted[:, index_top]
    clusters_col_sorted = clusters_col[index_top]

    tick_names = [RULE_NAME[k[0]] + " " + k[1] for k in keys]
    tick_names_sorted = [tick_names[i] for i in index_top]
    feature_names_labels = [t.rsplit(" ", 1)[0] for t in tick_names_sorted]

    epoch_binary = {}
    all_epochs = ["stim1", "stim2", "delay1", "delay2", "go1"]
    for e_name in all_epochs:
        epoch_binary[e_name] = [tick_names_sorted[i].rsplit(" ", 1)[-1] == e_name
                                for i in range(len(tick_names_sorted))]

    fig = plt.figure(figsize=(18, 9))
    plt.rcParams.update({"font.size": 13})
    hierarchy.set_link_color_palette(COLOR_PALETTE_FIG3)

    atlas_width = 0.55
    atlas_height = 0.52
    matrix_bottom = 0.12
    bar_height = 0.06
    dendro_top_bottom = matrix_bottom + atlas_height + 0.03
    thresh = 4

    axdendro_top = fig.add_axes([0.06, dendro_top_bottom, atlas_width - 0.05, 0.06])
    Z_top = dendrogram(Y_col, orientation="top", leaf_font_size=9,
                       color_threshold=thresh, above_threshold_color="gray",
                       labels=clusters_col_sorted)
    cluster_size, mid_top = make_cluster_midpoint_labels_fig3(clusters_col_sorted)
    for xi in range(len(mid_top) - 1):
        if cluster_size[xi] == 1:
            c = "k"
        else:
            color_ind = xi - np.sum(cluster_size[:xi] == 1)
            c = COLOR_PALETTE_FIG3[int(color_ind) % len(COLOR_PALETTE_FIG3)]
        clust_mid = mid_top[xi]
        rect = mpatches.Rectangle(
            (5 + 10 * (clust_mid - cluster_size[xi] / 2), -5),
            cluster_size[xi] * 10, 10,
            fill=True, color=c, facecolor=c, alpha=0.1, clip_on=False)
        axdendro_top.add_patch(rect)
        axdendro_top.text(clust_mid * 10, -3, str(int(clusters_col_sorted[int(clust_mid)])),
                          color=c, fontweight="bold", rotation=90, fontsize=9)
    axdendro_top.set_xticks([])
    axdendro_top.set_yticks([])
    for spine in axdendro_top.spines.values():
        spine.set_visible(False)

    axdendro = fig.add_axes([0.0, matrix_bottom, 0.04, atlas_height * 0.98])
    Z_left = dendrogram(Y_row, orientation="left", leaf_font_size=9,
                        color_threshold=2.07, above_threshold_color="gray",
                        labels=clusters_row_sorted)
    cluster_size, mid = make_cluster_midpoint_labels_fig3(clusters_row_sorted)
    for xi in range(len(mid) - 1):
        ci = xi if xi <= 22 else xi - 1
        c = COLOR_PALETTE_FIG3[int(ci) % len(COLOR_PALETTE_FIG3)]
        if xi == 22:
            c = "k"
        clust_mid = mid[xi]
        rect = mpatches.Rectangle(
            (-5, 10 + 10 * (clust_mid - cluster_size[xi] / 2)),
            5, cluster_size[xi] * 10,
            fill=True, color=c, facecolor=c, alpha=0.1, clip_on=False)
        axdendro.add_patch(rect)
        axdendro.text(-1.5, 10 * clust_mid - 20,
                      cluster_letter_label_fig3(clusters_row_sorted[int(clust_mid)]),
                      color=c, rotation=90, fontsize=9)
    axdendro.set_xticks([])
    axdendro.set_yticks([])
    for spine in axdendro.spines.values():
        spine.set_visible(False)

    axmatrix = fig.add_axes([0.06, matrix_bottom, atlas_width - 0.05, atlas_height])
    im = axmatrix.imshow(D_sorted[-1::-1, :], cmap="viridis", aspect="auto")
    axmatrix.set_xticks(range(len(tick_names_sorted)))
    axmatrix.set_xticklabels([])
    axmatrix.set_yticks([])
    axmatrix.set_ylim((len(D_sorted) + 1, 0))
    for spine in axmatrix.spines.values():
        spine.set_visible(False)

    axbars = fig.add_axes([0.06, matrix_bottom - bar_height - 0.02,
                           atlas_width - 0.05, bar_height])
    plot_epoch_rects_fig3(axbars, epoch_binary, all_epochs, n_cols=len(tick_names_sorted),
                          rect_height=bar_height / len(all_epochs), y_base=0.0)

    axlabels = fig.add_axes([0.06, matrix_bottom - bar_height - 0.085,
                             atlas_width - 0.05, 0.06])
    axlabels.set_xlim(-0.5, len(tick_names_sorted) - 0.5)
    axlabels.set_ylim(0, 1)
    axlabels.set_xticks(range(len(tick_names_sorted)))
    axlabels.set_xticklabels(feature_names_labels, fontsize=9, rotation=90, ha="center")
    axlabels.set_yticks([])
    for spine in axlabels.spines.values():
        spine.set_visible(False)

    fig.text(0.06 + (atlas_width - 0.05) / 2, 0.02,
             r"$\bf{Task\ Periods}$" + " (sorted by row similarity)",
             ha="center", fontsize=12)

    out_path = os.path.join(fig_dir, "fig3_A.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()
    return D_sorted, tick_names_sorted


def make_pca_h_cat(states_a, epochs_a, states_b, epochs_b, period, n_components=3):
    s_a, e_a = get_epoch_slice(epochs_a, period, states_a.shape[1])
    s_b, e_b = get_epoch_slice(epochs_b, period, states_b.shape[1])
    X_a = states_a[:, s_a:e_a, :].reshape(-1, states_a.shape[-1])
    X_b = states_b[:, s_b:e_b, :].reshape(-1, states_b.shape[-1])
    X = np.concatenate([X_a, X_b], axis=0)
    pca = PCA(n_components=n_components).fit(X)
    return pca.components_.T


def seed_states_for_period_fig3(states, epochs, period):
    T = states.shape[1]
    if period == "fix1":
        s_prev, e_prev = get_epoch_slice(epochs, "delay1", T)
    elif period == "stim1":
        s_prev, e_prev = get_epoch_slice(epochs, "fix1", T)
    elif period == "delay1":
        s_prev, e_prev = get_epoch_slice(epochs, "stim1", T)
    elif period == "stim2":
        s_prev, e_prev = get_epoch_slice(epochs, "delay1", T)
    elif period == "delay2":
        s_prev, e_prev = get_epoch_slice(epochs, "stim2", T)
    elif period == "go1":
        s_prev, e_prev = get_epoch_slice(epochs, "delay2", T)
    else:
        s_prev, e_prev = 0, T
    return states[:, e_prev - 1, :]


def compute_rule_interp_fps_fig3(model, inputs_a, states_a, epochs_a,
                                 inputs_b, states_b, epochs_b,
                                 rule_a, rule_b, period,
                                 n_interp=20, params=None):
    params = params or {}
    seed_a = seed_states_for_period_fig3(states_a, epochs_a, period)
    seed_b = seed_states_for_period_fig3(states_b, epochs_b, period)
    seed = np.concatenate([seed_a, seed_b], axis=0)

    s, _ = get_epoch_slice(epochs_a, period, states_a.shape[1])
    u_base = inputs_a[:, s, :].mean(axis=0).copy()
    u_base = period_input_with_rule(u_base, rule_a)

    defaults = {
        "n_candidates": 400,
        "n_iters": 5000,
        "lr": 1e-3,
        "noise_scale": 0.01,
        "q_thresh": 5e-2,
        "dedup_tol": 8e-2,
    }
    defaults.update(params)

    alphas = np.linspace(0, 1, n_interp)
    fp_by_alpha = []
    for alpha in alphas:
        u = u_base.copy()
        u[RULE_START:RULE_START + N_RULE_FIG3] = 0.0
        u[RULE_START + RULE_INDEX_MAP[rule_a]] = 1.0 - alpha
        u[RULE_START + RULE_INDEX_MAP[rule_b]] = alpha

        idx = np.random.choice(len(seed), size=defaults["n_candidates"], replace=True)
        z0 = seed[idx] + np.random.randn(defaults["n_candidates"], seed.shape[-1]) * defaults["noise_scale"]
        z0 = np.maximum(z0, 0.0)

        finder = NumericFixedPointFinder(
            n_candidates=defaults["n_candidates"], n_iters=defaults["n_iters"],
            lr=defaults["lr"], speed_tol=5e-1, dedup_tol=defaults["dedup_tol"],
            init_scale=0.5, init_positive=True,
        )
        fps = finder.find(
            model,
            task_input=torch.from_numpy(u).float().to(DEVICE),
            init_states=torch.from_numpy(z0).float().to(DEVICE),
        )
        fp_by_alpha.append((alpha, fps.points))
    return alphas, fp_by_alpha


def plot_fig3_de_pair(model, rule_a, rule_b, period,
                      q_thresh=5e-2, plot_unstable=True,
                      label=None, fig_dir=FIG_DIR):
    if label is None:
        label = {"dmcgo": "D", "delaydm1": "E"}.get(rule_a, label)
    print(f"\n=== Figure 3{label}: {rule_a} vs {rule_b} ({period}) ===")

    inputs_a, targets_a, mask_a, conds_a = generate_trials(
        rule_a, mode="test", sigma_x=0.0, seed=SEED)
    outputs_a, states_a = run_forward(model, inputs_a)
    epochs_a = conds_a[0]["epochs"]

    inputs_b, targets_b, mask_b, conds_b = generate_trials(
        rule_b, mode="test", sigma_x=0.0, seed=SEED)
    outputs_b, states_b = run_forward(model, inputs_b)
    epochs_b = conds_b[0]["epochs"]

    master_out = np.array([c["response_loc"] for c in conds_a])
    temp_out = np.array([c["response_loc"] for c in conds_b])
    align_inds = align_trials_by_output(master_out, temp_out)
    inputs_b = inputs_b[align_inds]
    targets_b = targets_b[align_inds]
    states_b = states_b[align_inds]

    D_use = make_pca_h_cat(states_a, epochs_a, states_b, epochs_b, period)
    D_use[:, 0] *= -1
    D_use[:, 1] *= -1

    cache_path = fp_interp_cache_path(
        MODEL_DIR, rule_a, rule_b, period, {"q_thresh": q_thresh}, 20, SEED)
    if os.path.exists(cache_path):
        alphas, fp_by_alpha = load_pickle(cache_path)
    else:
        alphas, fp_by_alpha = compute_rule_interp_fps_fig3(
            model, inputs_a, states_a, epochs_a,
            inputs_b, states_b, epochs_b,
            rule_a, rule_b, period, n_interp=20,
            params={"q_thresh": q_thresh})
        save_pickle((alphas, fp_by_alpha), cache_path)

    print(f"  Fixed points per alpha: " +
          ", ".join(str(len(fps)) for _, fps in fp_by_alpha))

    fig, ax = plt.subplots(figsize=(7, 7), tight_layout=True, facecolor="white")
    plt.rcParams.update({"font.size": 20})

    cmap_grad = cm.get_cmap("plasma")
    cmap_discrete = ["w", "k"]

    for ri, (rule, states, epochs, inputs) in enumerate(
            [(rule_a, states_a, epochs_a, inputs_a),
             (rule_b, states_b, epochs_b, inputs_b)]):
        s, e = get_epoch_slice(epochs, period, states.shape[1])
        x_epoch = states[:, s:e, :]

        n_trials = x_epoch.shape[0]
        trial_set = list(range(0, n_trials, max(n_trials // 8, 1)))
        stim_angles = decode_stim1_angle_fig3(inputs, epochs)

        for idx in trial_set:
            traj = x_epoch[idx] @ D_use[:, :2]
            c = cm.hsv(stim_angles[idx] / (2 * np.pi))
            ax.plot(traj[:, 0], traj[:, 1], "-", color=c,
                    linewidth=8, alpha=0.5)
            ax.plot(traj[-1, 0], traj[-1, 1], "^", color=c,
                    markersize=10, alpha=0.7)
            ax.plot(traj[0, 0], traj[0, 1], "x", color=c,
                    markersize=10, alpha=0.7)

        for idx in trial_set:
            traj = x_epoch[idx] @ D_use[:, :2]
            ax.plot(traj[:, 0], traj[:, 1], "-", color=cmap_discrete[ri],
                    linewidth=3.5, alpha=0.9)

    for alpha, fps in fp_by_alpha:
        c = cmap_grad(alpha)
        for fp in fps:
            proj = fp.z @ D_use[:, :2]
            if fp.is_stable:
                ax.plot(proj[0], proj[1], "o", color="w",
                        markeredgecolor=c, markeredgewidth=1,
                        markersize=4, alpha=0.4, zorder=0)
            elif plot_unstable:
                ax.plot(proj[0], proj[1], "o", color=c,
                        markerfacecolor="w", markeredgecolor=c,
                        markeredgewidth=1, markersize=4, alpha=0.4, zorder=0)

    x1, x2 = ax.get_xlim()
    y1, y2 = ax.get_ylim()
    ax.set_xlim([x1 - 0.2 * abs(x1), x2 + 0.2 * abs(x2)])
    ax.set_ylim([y1 - 0.2 * abs(y1), y2 + 0.2 * abs(y2)])

    epoch_label = period.replace("1", "").replace("2", "").capitalize()
    ax.set_xlabel(f"{epoch_label} State PC1", fontsize=18)
    ax.set_ylabel(f"{epoch_label} State PC2", fontsize=18)
    ax.set_title(f"{RULE_NAME[rule_a]} vs {RULE_NAME[rule_b]}\n{period} dynamics",
                 fontsize=16)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    out_path = os.path.join(fig_dir, f"fig3_{label}.png")
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    
    plt.show()



In [ ]:
_ = plot_fig3a(model_multi, FIG_DIR)


### 3.4 Figure 3D — shared category-memory attractors


The variance matrix predicts that `dmcgo` and `dmcnogo` share a category-memory motif. We interpolate the rule input between these two tasks during the `delay1` period and plot the trajectories and fixed points in a PCA basis built from both tasks' delay
states.  A continuous bridge of fixed points indicates a shared attractor.

In [ ]:
plot_fig3_de_pair(
    model_multi, "dmcgo", "dmcnogo", "delay1",
    q_thresh=5e-2, plot_unstable=False,
    label="D", fig_dir=FIG_DIR)

### 3.5 Figure 3E — shared continuous-memory ring attractors

Similarly, `delaydm1` and `delaydm2` both require memory of a continuous circular
variable.  We interpolate between them during the `delay2` period and recover a shared
ring attractor.

In [ ]:
plot_fig3_de_pair(
    model_multi, "delaydm1", "delaydm2", "delay2",
    q_thresh=5e-2, plot_unstable=True,
    label="E", fig_dir=FIG_DIR)

### 3.7 Figure 4B — task layout in context-endpoint space


For each task we collect the hidden state at the end of the fixation/context period
across many trials, then project the combined states onto their first two PCs.  Tasks
with similar stimulus computations cluster together in this context-endpoint space.

In [ ]:
def unit_vector_fig4(v):
    v = np.asarray(v, dtype=np.float64)
    norm = np.linalg.norm(v)
    if norm == 0:
        return v
    return v / norm


def angle_between_fig4(v1, v2):
    v1_u = unit_vector_fig4(v1)
    v2_u = unit_vector_fig4(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))


def same_stim_trial_fig4(master_inputs, task_num):
    inputs = master_inputs.copy()
    inputs[:, :, RULE_START_FIG4:RULE_START_FIG4 + N_RULE_FIG4] = 0.0
    inputs[:, :, RULE_START_FIG4 + task_num] = 1.0

    if task_num in REACT_TASK_INDICES_FIG4:
        inputs[:, :, 0] = 1.0

    return inputs


def make_h_combined_fig4(model, task_nums, trial_set, epoch, ind=-1, seed=0):
    master_inputs, _, _, master_conds = generate_trials(
        'delaygo', mode='random', n_trials=100, sigma_x=0.01, seed=seed,
    )
    epochs_master = master_conds[0]["epochs"]
    T = master_inputs.shape[1]

    if epoch == "stim1" and ind == 1:
        s, e = get_epoch_slice(epochs_master, "stim1", T)
        if s == e or s >= T:
            s, _ = get_epoch_slice(epochs_master, "go1", T)
            t_idx = s
        else:
            t_idx = s + ind
            if t_idx >= e:
                t_idx = s
    else:
        s, e = get_epoch_slice(epochs_master, epoch, T)
        if ind < 0:
            t_idx = e + ind
        else:
            t_idx = s + ind
        t_idx = max(0, min(t_idx, T - 1))

    h_combined = []
    for task_num in task_nums:
        inputs_task = same_stim_trial_fig4(master_inputs, task_num)
        _, states_task = run_forward(model, inputs_task)
        h_task = states_task[trial_set, t_idx, :]
        h_combined.append(h_task)

    return np.concatenate(h_combined, axis=0)


def plot_task_layout_PCA_fig4(h_combined, task_nums, color_set=COLOR_SET_FIG4,
                              pcs=(0, 1), figsize=7):
    n_trials_per_task = h_combined.shape[0] // len(task_nums)

    X_train = h_combined[0::2, :]
    pca = PCA(n_components=4)
    pca.fit(X_train)
    X_test = h_combined[1::2, :]
    D_use = pca.transform(X_test)[:, list(pcs)]

    fig = plt.figure(figsize=(figsize, figsize), tight_layout=True, facecolor='white')
    plt.rcParams.update({'font.size': 16})
    ax1 = plt.subplot(1, 1, 1)

    together = []
    offset = 0
    for t_num in task_nums:
        task_start = t_num * n_trials_per_task
        task_end = (t_num + 1) * n_trials_per_task
        odd_mask = (np.arange(task_start, task_end) % 2) == 1
        n_kept = odd_mask.sum()
        D_task = D_use[offset:offset + n_kept, :]
        offset += n_kept

        c = color_set[t_num]
        ax1.plot(D_task[:, 0], D_task[:, 1], 'o', c=c, alpha=0.2, markersize=10)
        rule_display = RULE_SET_NAMES_FIG4[t_num]
        together.append((np.mean(D_task[:, 0]), np.mean(D_task[:, 1]), rule_display, c))

    texts = []
    for x, y, s, c in together:
        texts.append(ax1.text(x, y, s, color=c, fontsize=10, weight="bold"))

    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.set_xticks([])
    ax1.set_yticks([])
    ax1.set_xlabel('Context Endpt. State PC1', fontsize=20)
    ax1.set_ylabel('Context Endpt. State PC2', fontsize=20)
    adjust_text(texts, only_move={'points': 'y', 'texts': 'y'},
                arrowprops=dict(arrowstyle="->", color='k', lw=1))

    out_path = os.path.join(FIG_DIR, "fig4_B.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return together


def make_angle_mat_fig4(X):
    X = np.squeeze(X).astype(np.float64)
    n = X.shape[0]
    theta_mat = np.zeros((n, n))
    for xi in range(n):
        v1 = X[xi, :]
        for yi in range(n):
            v2 = X[yi, :]
            theta_mat[xi, yi] = 180.0 * angle_between_fig4(v1, v2) / np.pi
    return theta_mat


def make_dst_mat_fig4(X):
    X = np.squeeze(X).astype(np.float64)
    n = X.shape[0]
    dst = np.zeros((n, n))
    for xi in range(n):
        for yi in range(n):
            dst[xi, yi] = distance.euclidean(X[xi, :], X[yi, :])
    return dst


def make_sub_mat_fig4(D_concat, T_concat, motif_grp_concat,
                      D_mat, T_mat, labels, color_set=COLOR_SET_FIG4):
    D_sub = D_mat[labels, :][:, labels]
    T_sub = T_mat[labels, :][:, labels]

    same_motif = np.zeros((len(color_set), len(color_set)), dtype=bool)
    for i in range(len(color_set)):
        for j in range(len(color_set)):
            same_motif[i, j] = color_set[i] == color_set[j]
    motif_grp = same_motif[labels, :][:, labels]

    D_sub = np.tril(D_sub, k=-1)
    T_sub = np.tril(T_sub, k=-1)
    motif_grp = np.tril(motif_grp, k=-1)

    mask = D_sub > 0
    if len(D_concat) == 0:
        D_concat = D_sub[mask][:, np.newaxis]
        T_concat = T_sub[mask][:, np.newaxis]
        motif_grp_concat = motif_grp[mask][:, np.newaxis]
    else:
        D_concat = np.concatenate((D_concat, D_sub[mask][:, np.newaxis]), axis=1)
        T_concat = np.concatenate((T_concat, T_sub[mask][:, np.newaxis]), axis=1)
        motif_grp_concat = np.concatenate(
            (motif_grp_concat, motif_grp[mask][:, np.newaxis]), axis=1)
    return D_concat, T_concat, motif_grp_concat


def plot_fig4c(h_combined_fix, h_combined_stim, task_nums,
               color_set=COLOR_SET_FIG4, figsize=7):
    h_stim_diff = h_combined_stim - h_combined_fix
    theta_mat = make_angle_mat_fig4(h_stim_diff)
    dst_fix = make_dst_mat_fig4(h_combined_fix)

    n_trials_per_task = h_combined_fix.shape[0] // len(task_nums)
    batch_size = n_trials_per_task

    D_concat = {k: np.array([]).reshape(0, 0) for k in TASK_LABELS_FIG4.keys()}
    T_concat = {k: np.array([]).reshape(0, 0) for k in TASK_LABELS_FIG4.keys()}
    motif_grp_concat = {k: np.array([]).reshape(0, 0) for k in TASK_LABELS_FIG4.keys()}

    for xi in range(batch_size):
        dst_fix_stim = dst_fix[xi::batch_size, :][:, xi::batch_size]
        theta_mat_stim = theta_mat[xi::batch_size, :][:, xi::batch_size]

        for k, labels in TASK_LABELS_FIG4.items():
            D_concat[k], T_concat[k], motif_grp_concat[k] = make_sub_mat_fig4(
                D_concat[k], T_concat[k], motif_grp_concat[k],
                dst_fix_stim, theta_mat_stim, labels, color_set=color_set,
            )

    fig = plt.figure(figsize=(figsize, figsize), tight_layout=True, facecolor='white')
    plt.rcParams.update({'font.size': 20})
    ax1 = plt.subplot(1, 1, 1)

    D_concat_out = {}
    T_concat_out = {}
    for k in TASK_LABELS_FIG4.keys():
        x = np.mean(D_concat[k], axis=1)
        y = np.mean(T_concat[k], axis=1)
        D_concat_out[k] = x
        T_concat_out[k] = y
        if k == 'all':
            ax1.scatter(x, y, c='k', alpha=0.5, s=30)
        else:
            ax1.plot(x, y, 'ok', markerfacecolor='none', markersize=25,
                     alpha=1, label=k, linewidth=5)
            ax1.text(x, y, k, fontweight='bold')

    ax1.set_xlabel('Distance between Initial Conditions')
    ax1.set_ylabel('Angle between First Step \n of Trajectories (deg.)')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    out_path = os.path.join(FIG_DIR, "fig4_C.png")
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    
    plt.show()
    return D_concat_out, T_concat_out


def save_fig4b_csv(together):
    csv_path = os.path.join(FIG_DIR, "fig4b.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["PC1", "PC2", "Rule Name", "Color"])
        for item in together:
            pc1, pc2, rule_name, color = item
            writer.writerow([pc1, pc2, rule_name, color])


def save_fig4c_csv(D_concat_out, T_concat_out):
    csv_path = os.path.join(FIG_DIR, "fig4c.csv")
    with open(csv_path, "w", newline="") as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["Task", "Mean Distance", "Mean Angle"])
        for k in TASK_LABELS_FIG4.keys():
            mean_distance = D_concat_out[k]
            mean_angle = T_concat_out[k]
            writer.writerow([k, mean_distance, mean_angle])


In [ ]:
h_combined_fix = make_h_combined_fig4(
    model_multi, list(range(15)), list(range(80)), 'fix1', ind=-1, seed=SEED)
print(f"h_combined_fix shape: {h_combined_fix.shape}")

together_fig4 = plot_task_layout_PCA_fig4(h_combined_fix, list(range(15)), figsize=6)
save_fig4b_csv(together_fig4)

### 3.8 Figure 4C — distance between initial conditions vs first-step angle


For pairs of trials from different tasks with the same stimulus, we measure (1) the
Euclidean distance between their context-endpoint states and (2) the angle between
their state-update directions on the first stimulus timestep.  Tasks that share a
stimulus-period motif fall in the lower-left corner (nearby initial conditions and
aligned trajectories).

In [ ]:
trial_set_c = list(range(0, 80, 10))
h_combined_fix_c = make_h_combined_fig4(
    model_multi, list(range(15)), trial_set_c, 'fix1', ind=-1, seed=SEED)
h_combined_stim_c = make_h_combined_fig4(
    model_multi, list(range(15)), trial_set_c, 'stim1', ind=1, seed=SEED)
print(f"h_combined_fix_c shape: {h_combined_fix_c.shape}")
print(f"h_combined_stim_c shape: {h_combined_stim_c.shape}")

D_concat_out, T_concat_out = plot_fig4c(
    h_combined_fix_c, h_combined_stim_c, list(range(15)))
save_fig4c_csv(D_concat_out, T_concat_out)

## Summary

This notebook has walked through the three main analyses of Driscoll et al. (2024):

1. **Single-task ring attractor** (Figure 1): a MemoryPro network stores the stimulus
   on a ring attractor during the delay and rotates it into output-potent space during
   the response.
2. **Two-task shared fixed points** (Figure 2): MemoryPro and MemoryAnti share the
   same ring attractor; the rule input shifts fixed-point locations to flip the
   response direction.
3. **15-task dynamical motifs** (Figures 3–4): a variance-matrix atlas identifies
   clusters of units that implement reusable motifs, and rule-input interpolation
   confirms that similar tasks share attractors.

All figures are saved under `figs/13/`.